# 05 — NLP / RAG: Drug Evidence Pipeline

**P.U.L.S.E.**

Demonstrates the full RAG (Retrieval-Augmented Generation) pipeline:

1. Drug review dataset EDA (4,143 reviews, 2 sources)
2. ChromaDB vector index exploration (11 k chunks)
3. Embedding space visualisation via UMAP/PCA
4. End-to-end retrieval + Mistral 7B synthesis
5. No-LLM fallback comparison

In [ ]:
import os, sys
ROOT = os.path.dirname(os.path.abspath('.'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
print('Ready')

## 1  Drug review dataset EDA

In [ ]:
from src.data.loader import load_drug_reviews

df = load_drug_reviews()
print(f'Reviews: {len(df):,}  |  columns: {list(df.columns)}')
df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rating distribution
df['rating'].value_counts().sort_index().plot.bar(
    ax=axes[0], color=sns.color_palette('muted')[0]
)
axes[0].set_title('Rating Distribution (1-10)')
axes[0].set_xlabel('Rating')

# Top conditions
top_c = df['condition'].value_counts().head(12)
top_c.plot.barh(ax=axes[1], color=sns.color_palette('muted')[1])
axes[1].set_title('Top 12 Conditions')
axes[1].invert_yaxis()

# Review text length
df['review_len'] = df['benefitsReview'].str.len().fillna(0)
axes[2].hist(df['review_len'], bins=40, color=sns.color_palette('muted')[2], edgecolor='white')
axes[2].set_title('Benefits Review Length (chars)')
axes[2].set_xlabel('Characters')

plt.suptitle('Drug Review Dataset EDA', fontsize=14)
plt.tight_layout()
plt.show()

## 2  ChromaDB vector index stats

In [ ]:
from src.nlp.rag_pipeline import load_drug_index

vs = load_drug_index()
count = vs._collection.count()
print(f'Vectors in index: {count:,}')

# Sample 5 docs
sample = vs.similarity_search('diabetes medication benefits', k=5)
for i, doc in enumerate(sample):
    m = doc.metadata
    print(f'[{i+1}] {m["drug"]} ({m["condition"]}) rating={m["rating"]:.0f}/10')
    print(f'     {doc.page_content[:100]} ...')

## 3  Embedding space — PCA 2D projection

In [ ]:
from sklearn.decomposition import PCA
from langchain_huggingface import HuggingFaceEmbeddings

# Sample 300 reviews for visualisation
sample_docs = vs.similarity_search('patient experience', k=300)
texts   = [d.page_content[:200] for d in sample_docs]
labels  = [d.metadata.get('condition', 'unknown')[:20] for d in sample_docs]
ratings = [d.metadata.get('rating', 5) for d in sample_docs]

embed_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectors = np.array(embed_model.embed_documents(texts))

pca  = PCA(n_components=2, random_state=42)
proj = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(proj[:, 0], proj[:, 1], c=ratings,
                cmap='RdYlGn', s=20, alpha=0.7, vmin=1, vmax=10)
plt.colorbar(sc, ax=ax, label='Patient Rating (1-10)')
ax.set_title('Drug Review Embedding Space — PCA 2D (colour = rating)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

## 4  End-to-end RAG query — Retrieval Only (fast)

In [ ]:
from src.nlp.summarizer import summarize_drug_no_llm

result = summarize_drug_no_llm(
    condition='depression',
    drug='Lexapro',
    top_k=8,
)

print(result['summary'])
print()
print(f'Sources: {len(result["source_docs"])} chunks')

## 5  End-to-end RAG query — Full Mistral 7B synthesis

In [ ]:
from src.nlp.summarizer import summarize_drug

# NOTE: requires Ollama running locally: ollama pull mistral
result = summarize_drug(
    condition='Type 2 Diabetes',
    drug='Metformin',
    top_k=8,
)

print(result['summary'])

## 6  Rating by condition — boxplot

In [ ]:
top_conditions = df['condition'].value_counts().head(8).index
df_top = df[df['condition'].isin(top_conditions)]

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_top, x='condition', y='rating', palette='muted', ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.set_title('Drug Ratings by Condition (top 8)')
ax.set_ylabel('Patient Rating (1-10)')
plt.tight_layout()
plt.show()